In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,mean_absolute_percentage_error
from sklearn.model_selection import KFold

data=pd.read_excel(r'D:\AI4polymer\cocrystal design\Data-0320\deltaML result\deltaML4DFT_rawMD_merged_data_0512.xlsx')
data_PTT = data.dropna(subset=['photothermal'])
data_PTT.reset_index(drop=True, inplace=True)
data_PDT = data.dropna(subset=['photodynamic'])
data_PDT.reset_index(drop=True, inplace=True)

In [2]:
data_PTT=data_PTT.dropna()
data_PTT.reset_index(drop=True, inplace=True)
data_PDT=data_PDT.dropna()
data_PDT.reset_index(drop=True, inplace=True)

## 1. PTT

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
data_PTT["PTT_bin"] = data_PTT["photothermal"].astype(str).replace({"middle": "weak"})
data_PTT['PTT_bin'].value_counts()

PTT_bin
strong    129
weak      114
Name: count, dtype: int64

In [ ]:
DFT_cols =[
    'es1_ev_high_pred',
    'fs1_high_pred',
    'et1_ev_high_pred',
    'delta_est_ev_high_pred',
    'min_abs_s1_tn_ev_high_pred',
    'd_index_ang_high_pred',
    't_index_ang_high_pred',
    'zpe_ha_pred',
    'gcorr_ha_pred',
    'alpha_iso_bohr3_pred',
    's_vib_cal_molk_pred',
    'svib_frac_pred',
    'hlg_kjmol_high_pred',
    'dipole_gs_debye_high_pred',
    'ci_index',]

MD_cols=['vdw_energy_kjmol','total_energy_kjmol','rg_center_norm','rdf_first_peak_height', 'hbond_dynamic', 'hbond_static']

random_seed = 212
np.random.seed(random_seed)

### 1.1 DFT+MD

In [6]:
### PTT DFT+MD
feature_cols = DFT_cols + MD_cols

target_col = 'PTT_bin'

# 只保留实际存在的列
existing_feature_cols = [col for col in feature_cols if col in data_PTT.columns]
missing_cols = [col for col in feature_cols if col not in data_PTT.columns]

print("存在的特征列数:", len(existing_feature_cols))
print("缺失的列:", missing_cols)

# 提取数据
X = data_PTT[existing_feature_cols].copy()
y = data_PTT[target_col].copy()

# 2. 自动去掉非数值列
# =========================
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    print("\n以下列不是数值列，将被移除：")
    print(non_numeric_cols)

X = X.select_dtypes(include=[np.number]).copy()

print("\n最终用于建模的特征数:", X.shape[1])
print("最终特征列:")
print(X.columns.tolist())

# =========================
# 3. 标签编码
# strong -> 1, weak -> 0
# =========================
y = y.map({'strong': 1, 'weak': 0})

# 检查是否有未映射成功的标签
if y.isnull().any():
    raise ValueError("PTT_bin 中存在不是 'strong' 或 'weak' 的标签，请检查数据。")

print("\n标签分布：")
print(y.value_counts())
# =========================
# 4. 定义五折分层交叉验证
# =========================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_seed)

# =========================
# 5. 定义模型管道
# =========================
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=random_seed,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

# =========================
# 6. 交叉验证评估
# =========================
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    pipeline,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("\n===== 5-Fold Cross Validation Results =====")
for metric in scoring.keys():
    scores = cv_results[f'test_{metric}']
    print(f"{metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")

# =========================
# 7. 获取交叉验证整体预测结果
# =========================
y_pred = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict',
    n_jobs=-1
)

y_proba = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# =========================
# 8. 输出整体指标
# =========================
print("\n===== Overall CV Prediction Metrics =====")
print("Accuracy :", accuracy_score(y, y_pred))
print("Precision:", precision_score(y, y_pred))
print("Recall   :", recall_score(y, y_pred))
print("F1-score :", f1_score(y, y_pred))
print("ROC-AUC  :", roc_auc_score(y, y_proba))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y, y_pred))

print("\n===== Classification Report =====")
print(classification_report(y, y_pred, target_names=['weak', 'strong']))

# ====================================================================================================
# 7. 模型调参 + 十折分层交叉验证
# ====================================================================================================
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# =========================
# 7.1 十折分层交叉验证设置
# =========================
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_seed
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# 调参时以 F1-score 作为主指标
refit_metric = "f1"


# =========================
# 7.2 类别不平衡处理
# =========================
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)

if n_pos == 0 or n_neg == 0:
    raise ValueError("y 中必须同时包含 strong 和 weak 两类。")

scale_pos_weight = n_neg / n_pos

print("\n类别分布:")
print(y.value_counts())
print(f"XGBoost scale_pos_weight: {scale_pos_weight:.4f}")


# =========================
# 7.3 定义模型和较小的参数搜索空间
# =========================
models_and_params = {
    "RandomForest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                random_state=random_seed,
                n_jobs=-1,
                class_weight="balanced"
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", 0.5]
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=random_seed,
                n_jobs=-1,
                scale_pos_weight=scale_pos_weight
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
        }
    },

    "MLP": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                random_state=random_seed,
                max_iter=1000,
                early_stopping=True
            ))
        ]),
        "param_grid": {
            "model__hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "model__alpha": [1e-4, 1e-3],
            "model__learning_rate_init": [1e-3, 5e-4]
        }
    },

    "LogisticRegression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=random_seed,
                class_weight="balanced",
                max_iter=2000
            ))
        ]),
        "param_grid": {
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear"]
        }
    }
}


# =========================
# 7.4 对每个模型进行 GridSearchCV
# =========================
best_estimators = {}
summary_results = []

for model_name, item in models_and_params.items():

    print("\n" + "=" * 80)
    print(f"Start Grid Search for: {model_name}")
    print("=" * 80)

    grid_search = GridSearchCV(
        estimator=item["pipeline"],
        param_grid=item["param_grid"],
        scoring=scoring,
        refit=refit_metric,
        cv=skf,
        n_jobs=-1,
        return_train_score=False
    )

    grid_search.fit(X, y)

    best_estimators[model_name] = grid_search.best_estimator_

    print(f"\nBest parameters for {model_name}:")
    print(grid_search.best_params_)

    print(f"\nBest mean CV {refit_metric}: {grid_search.best_score_:.4f}")

    best_idx = grid_search.best_index_

    result_row = {
        "Model": model_name,
        "Best_Params": grid_search.best_params_
    }

    for metric in scoring.keys():
        mean_score = grid_search.cv_results_[f"mean_test_{metric}"][best_idx]
        std_score = grid_search.cv_results_[f"std_test_{metric}"][best_idx]

        result_row[f"{metric}_mean"] = mean_score
        result_row[f"{metric}_std"] = std_score

        print(f"{metric:10s}: {mean_score:.4f} ± {std_score:.4f}")

    summary_results.append(result_row)


# =========================
# 7.5 汇总不同模型的调参结果
# =========================
summary_df = pd.DataFrame(summary_results)

display_cols = [
    "Model",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std",
    "roc_auc_mean", "roc_auc_std",
    "Best_Params"
]

summary_df = summary_df[display_cols]

print("\n" + "=" * 80)
print("Grid Search Summary")
print("=" * 80)
print(summary_df)


# =========================
# 8. 使用最佳模型进行整体交叉验证预测
# =========================
overall_results = []

for model_name, best_model in best_estimators.items():

    print("\n" + "=" * 80)
    print(f"Overall CV Prediction Metrics: {model_name}")
    print("=" * 80)

    y_pred = cross_val_predict(
        best_model,
        X,
        y,
        cv=skf,
        method="predict",
        n_jobs=-1
    )

    # 大多数模型都有 predict_proba
    if hasattr(best_model, "predict_proba"):
        y_proba = cross_val_predict(
            best_model,
            X,
            y,
            cv=skf,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
    else:
        y_proba = None

    acc = accuracy_score(y, y_pred)
    pre = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    if y_proba is not None:
        auc = roc_auc_score(y, y_proba)
    else:
        auc = np.nan

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y,
        y_pred,
        target_names=["weak", "strong"]
    ))

    overall_results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    })


# =========================
# 8.1 整体预测结果汇总
# =========================
overall_results_df = pd.DataFrame(overall_results)

print("\n" + "=" * 80)
print("Overall CV Prediction Summary")
print("=" * 80)
print(overall_results_df)


# =========================
# 8.2 选择最优模型
# =========================
best_model_name = overall_results_df.sort_values(
    by="F1",
    ascending=False
).iloc[0]["Model"]

best_model = best_estimators[best_model_name]

print("\n" + "=" * 80)
print(f"Best model based on overall CV F1-score: {best_model_name}")
print("=" * 80)

print("\nBest model pipeline:")
print(best_model)


df_dft_md_best_overall = overall_results_df[
    overall_results_df["Model"] == best_model_name
].copy()

df_dft_md_best_summary = pd.DataFrame({
    "method": ["DFT_MD"],
    "best_model": [best_model_name],
    "test_accuracy": [f"{df_dft_md_best_overall['Accuracy'].iloc[0]:.4f}"],
    "test_precision": [f"{df_dft_md_best_overall['Precision'].iloc[0]:.4f}"],
    "test_recall": [f"{df_dft_md_best_overall['Recall'].iloc[0]:.4f}"],
    "test_f1": [f"{df_dft_md_best_overall['F1'].iloc[0]:.4f}"],
    "test_roc_auc": [f"{df_dft_md_best_overall['ROC_AUC'].iloc[0]:.4f}"]
})

存在的特征列数: 21
缺失的列: []

最终用于建模的特征数: 21
最终特征列:
['es1_ev_high_pred', 'fs1_high_pred', 'et1_ev_high_pred', 'delta_est_ev_high_pred', 'min_abs_s1_tn_ev_high_pred', 'd_index_ang_high_pred', 't_index_ang_high_pred', 'zpe_ha_pred', 'gcorr_ha_pred', 'alpha_iso_bohr3_pred', 's_vib_cal_molk_pred', 'svib_frac_pred', 'hlg_kjmol_high_pred', 'dipole_gs_debye_high_pred', 'ci_index', 'vdw_energy_kjmol', 'total_energy_kjmol', 'rg_center_norm', 'rdf_first_peak_height', 'hbond_dynamic', 'hbond_static']

标签分布：
PTT_bin
1    129
0    114
Name: count, dtype: int64

===== 5-Fold Cross Validation Results =====
accuracy  : 0.7600 ± 0.1093
precision : 0.7669 ± 0.0869
recall    : 0.7891 ± 0.1653
f1        : 0.7719 ± 0.1118
roc_auc   : 0.8270 ± 0.0944

===== Overall CV Prediction Metrics =====
Accuracy : 0.7613168724279835
Precision: 0.7669172932330827
Recall   : 0.7906976744186046
F1-score : 0.7786259541984732
ROC-AUC  : 0.8338093295253638

===== Confusion Matrix =====
[[ 83  31]
 [ 27 102]]

===== Classification R

### 1.2 DFT

In [10]:
##################################################  PTT DFT  ###################################################
feature_cols = DFT_cols

target_col = 'PTT_bin'

# 只保留实际存在的列
existing_feature_cols = [col for col in feature_cols if col in data_PTT.columns]
missing_cols = [col for col in feature_cols if col not in data_PTT.columns]

print("存在的特征列数:", len(existing_feature_cols))
print("缺失的列:", missing_cols)

# 提取数据
X = data_PTT[existing_feature_cols].copy()
y = data_PTT[target_col].copy()

# 2. 自动去掉非数值列
# =========================
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    print("\n以下列不是数值列，将被移除：")
    print(non_numeric_cols)

X = X.select_dtypes(include=[np.number]).copy()

print("\n最终用于建模的特征数:", X.shape[1])
print("最终特征列:")
print(X.columns.tolist())

# =========================
# 3. 标签编码
# strong -> 1, weak -> 0
# =========================
y = y.map({'strong': 1, 'weak': 0})

# 检查是否有未映射成功的标签
if y.isnull().any():
    raise ValueError("PTT_bin 中存在不是 'strong' 或 'weak' 的标签，请检查数据。")

print("\n标签分布：")
print(y.value_counts())
# =========================
# 4. 定义五折分层交叉验证
# =========================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_seed)

# =========================
# 5. 定义模型管道
# =========================
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=random_seed,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

# =========================
# 6. 交叉验证评估
# =========================
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    pipeline,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("\n===== 5-Fold Cross Validation Results =====")
for metric in scoring.keys():
    scores = cv_results[f'test_{metric}']
    print(f"{metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")

# =========================
# 7. 获取交叉验证整体预测结果
# =========================
y_pred = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict',
    n_jobs=-1
)

y_proba = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# =========================
# 8. 输出整体指标
# =========================
print("\n===== Overall CV Prediction Metrics =====")
print("Accuracy :", accuracy_score(y, y_pred))
print("Precision:", precision_score(y, y_pred))
print("Recall   :", recall_score(y, y_pred))
print("F1-score :", f1_score(y, y_pred))
print("ROC-AUC  :", roc_auc_score(y, y_proba))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y, y_pred))

print("\n===== Classification Report =====")
print(classification_report(y, y_pred, target_names=['weak', 'strong']))

dft_10cv_results = cv_results
df_dft_10cv = pd.DataFrame(dft_10cv_results)

metric_cols = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']
mean_series = df_dft_10cv[metric_cols].mean()
std_series = df_dft_10cv[metric_cols].std()

df_dft_summary = pd.DataFrame({
    col: [f"{mean_series[col]:.4f}±{std_series[col]:.4f}"]
    for col in metric_cols
})

df_dft_summary.insert(0, 'method', 'DFT')

################################################## ####################################################

# =========================
# 7. 模型调参 + 十折分层交叉验证
# =========================

# =========================
# 7.1 十折分层交叉验证设置
# =========================
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_seed
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# 调参时以 F1-score 作为主指标
refit_metric = "f1"


# =========================
# 7.2 类别不平衡处理
# =========================
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)

if n_pos == 0 or n_neg == 0:
    raise ValueError("y 中必须同时包含 strong 和 weak 两类。")

scale_pos_weight = n_neg / n_pos

print("\n类别分布:")
print(y.value_counts())
print(f"XGBoost scale_pos_weight: {scale_pos_weight:.4f}")


# =========================
# 7.3 定义模型和较小的参数搜索空间
# =========================
models_and_params = {
    "RandomForest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                random_state=random_seed,
                n_jobs=-1,
                class_weight="balanced"
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", 0.5]
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=random_seed,
                n_jobs=-1,
                scale_pos_weight=scale_pos_weight
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
        }
    },

    "MLP": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                random_state=random_seed,
                max_iter=1000,
                early_stopping=True
            ))
        ]),
        "param_grid": {
            "model__hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "model__alpha": [1e-4, 1e-3],
            "model__learning_rate_init": [1e-3, 5e-4]
        }
    },

    "LogisticRegression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=random_seed,
                class_weight="balanced",
                max_iter=2000
            ))
        ]),
        "param_grid": {
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear"]
        }
    }
}


# =========================
# 7.4 对每个模型进行 GridSearchCV
# =========================
best_estimators = {}
summary_results = []

for model_name, item in models_and_params.items():

    print("\n" + "=" * 80)
    print(f"Start Grid Search for: {model_name}")
    print("=" * 80)

    grid_search = GridSearchCV(
        estimator=item["pipeline"],
        param_grid=item["param_grid"],
        scoring=scoring,
        refit=refit_metric,
        cv=skf,
        n_jobs=-1,
        return_train_score=False
    )

    grid_search.fit(X, y)

    best_estimators[model_name] = grid_search.best_estimator_

    print(f"\nBest parameters for {model_name}:")
    print(grid_search.best_params_)

    print(f"\nBest mean CV {refit_metric}: {grid_search.best_score_:.4f}")

    best_idx = grid_search.best_index_

    result_row = {
        "Model": model_name,
        "Best_Params": grid_search.best_params_
    }

    for metric in scoring.keys():
        mean_score = grid_search.cv_results_[f"mean_test_{metric}"][best_idx]
        std_score = grid_search.cv_results_[f"std_test_{metric}"][best_idx]

        result_row[f"{metric}_mean"] = mean_score
        result_row[f"{metric}_std"] = std_score

        print(f"{metric:10s}: {mean_score:.4f} ± {std_score:.4f}")

    summary_results.append(result_row)


# =========================
# 7.5 汇总不同模型的调参结果
# =========================
summary_df = pd.DataFrame(summary_results)

display_cols = [
    "Model",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std",
    "roc_auc_mean", "roc_auc_std",
    "Best_Params"
]

summary_df = summary_df[display_cols]

print("\n" + "=" * 80)
print("Grid Search Summary")
print("=" * 80)
print(summary_df)


# =========================
# 8. 使用最佳模型进行整体交叉验证预测
# =========================
overall_results = []

for model_name, best_model in best_estimators.items():

    print("\n" + "=" * 80)
    print(f"Overall CV Prediction Metrics: {model_name}")
    print("=" * 80)

    y_pred = cross_val_predict(
        best_model,
        X,
        y,
        cv=skf,
        method="predict",
        n_jobs=-1
    )

    # 大多数模型都有 predict_proba
    if hasattr(best_model, "predict_proba"):
        y_proba = cross_val_predict(
            best_model,
            X,
            y,
            cv=skf,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
    else:
        y_proba = None

    acc = accuracy_score(y, y_pred)
    pre = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    if y_proba is not None:
        auc = roc_auc_score(y, y_proba)
    else:
        auc = np.nan

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y,
        y_pred,
        target_names=["weak", "strong"]
    ))

    overall_results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    })


# =========================
# 8.1 整体预测结果汇总
# =========================
overall_results_df = pd.DataFrame(overall_results)

print("\n" + "=" * 80)
print("Overall CV Prediction Summary")
print("=" * 80)
print(overall_results_df)


# =========================
# 8.2 选择最优模型
# =========================
best_model_name = overall_results_df.sort_values(
    by="F1",
    ascending=False
).iloc[0]["Model"]

best_model = best_estimators[best_model_name]

print("\n" + "=" * 80)
print(f"Best model based on overall CV F1-score: {best_model_name}")
print("=" * 80)

print("\nBest model pipeline:")
print(best_model)

df_dft_best_overall = overall_results_df[
    overall_results_df["Model"] == best_model_name
].copy()

df_dft_best_summary = pd.DataFrame({
    "method": ["DFT"],
    "best_model": [best_model_name],
    "test_accuracy": [f"{df_dft_best_overall['Accuracy'].iloc[0]:.4f}"],
    "test_precision": [f"{df_dft_best_overall['Precision'].iloc[0]:.4f}"],
    "test_recall": [f"{df_dft_best_overall['Recall'].iloc[0]:.4f}"],
    "test_f1": [f"{df_dft_best_overall['F1'].iloc[0]:.4f}"],
    "test_roc_auc": [f"{df_dft_best_overall['ROC_AUC'].iloc[0]:.4f}"]
})

df_PTT_best_summary = pd.concat([df_dft_md_best_summary, df_dft_best_summary], ignore_index=True)

存在的特征列数: 15
缺失的列: []

最终用于建模的特征数: 15
最终特征列:
['es1_ev_high_pred', 'fs1_high_pred', 'et1_ev_high_pred', 'delta_est_ev_high_pred', 'min_abs_s1_tn_ev_high_pred', 'd_index_ang_high_pred', 't_index_ang_high_pred', 'zpe_ha_pred', 'gcorr_ha_pred', 'alpha_iso_bohr3_pred', 's_vib_cal_molk_pred', 'svib_frac_pred', 'hlg_kjmol_high_pred', 'dipole_gs_debye_high_pred', 'ci_index']

标签分布：
PTT_bin
1    129
0    114
Name: count, dtype: int64

===== 5-Fold Cross Validation Results =====
accuracy  : 0.7435 ± 0.1248
precision : 0.7513 ± 0.1125
recall    : 0.7814 ± 0.1531
f1        : 0.7614 ± 0.1176
roc_auc   : 0.8275 ± 0.0949

===== Overall CV Prediction Metrics =====
Accuracy : 0.7448559670781894
Precision: 0.7481481481481481
Recall   : 0.7829457364341085
F1-score : 0.7651515151515151
ROC-AUC  : 0.8314973480212159

===== Confusion Matrix =====
[[ 80  34]
 [ 28 101]]

===== Classification Report =====
              precision    recall  f1-score   support

        weak       0.74      0.70      0.72       1

### 1.3 MD

In [12]:
##################################################  PTT MD  ###################################################
feature_cols = MD_cols

target_col = 'PTT_bin'

# 只保留实际存在的列
existing_feature_cols = [col for col in feature_cols if col in data_PTT.columns]
missing_cols = [col for col in feature_cols if col not in data_PTT.columns]

print("存在的特征列数:", len(existing_feature_cols))
print("缺失的列:", missing_cols)

# 提取数据
X = data_PTT[existing_feature_cols].copy()
y = data_PTT[target_col].copy()

# 2. 自动去掉非数值列
# =========================
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    print("\n以下列不是数值列，将被移除：")
    print(non_numeric_cols)

X = X.select_dtypes(include=[np.number]).copy()

print("\n最终用于建模的特征数:", X.shape[1])
print("最终特征列:")
print(X.columns.tolist())

# =========================
# 3. 标签编码
# strong -> 1, weak -> 0
# =========================
y = y.map({'strong': 1, 'weak': 0})

# 检查是否有未映射成功的标签
if y.isnull().any():
    raise ValueError("PTT_bin 中存在不是 'strong' 或 'weak' 的标签，请检查数据。")

print("\n标签分布：")
print(y.value_counts())
# =========================
# 4. 定义五折分层交叉验证
# =========================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_seed)

# =========================
# 5. 定义模型管道
# =========================
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=random_seed,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

# =========================
# 6. 交叉验证评估
# =========================
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    pipeline,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("\n===== 5-Fold Cross Validation Results =====")
for metric in scoring.keys():
    scores = cv_results[f'test_{metric}']
    print(f"{metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")

# =========================
# 7. 获取交叉验证整体预测结果
# =========================
y_pred = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict',
    n_jobs=-1
)

y_proba = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# =========================
# 8. 输出整体指标
# =========================
print("\n===== Overall CV Prediction Metrics =====")
print("Accuracy :", accuracy_score(y, y_pred))
print("Precision:", precision_score(y, y_pred))
print("Recall   :", recall_score(y, y_pred))
print("F1-score :", f1_score(y, y_pred))
print("ROC-AUC  :", roc_auc_score(y, y_proba))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y, y_pred))

print("\n===== Classification Report =====")
print(classification_report(y, y_pred, target_names=['weak', 'strong']))

dft_10cv_results = cv_results
df_dft_10cv = pd.DataFrame(dft_10cv_results)

metric_cols = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']
mean_series = df_dft_10cv[metric_cols].mean()
std_series = df_dft_10cv[metric_cols].std()

df_dft_summary = pd.DataFrame({
    col: [f"{mean_series[col]:.4f}±{std_series[col]:.4f}"]
    for col in metric_cols
})

df_dft_summary.insert(0, 'method', 'DFT')

################################################## ####################################################

# =========================
# 7. 模型调参 + 十折分层交叉验证
# =========================

# =========================
# 7.1 十折分层交叉验证设置
# =========================
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_seed
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# 调参时以 F1-score 作为主指标
refit_metric = "f1"


# =========================
# 7.2 类别不平衡处理
# =========================
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)

if n_pos == 0 or n_neg == 0:
    raise ValueError("y 中必须同时包含 strong 和 weak 两类。")

scale_pos_weight = n_neg / n_pos

print("\n类别分布:")
print(y.value_counts())
print(f"XGBoost scale_pos_weight: {scale_pos_weight:.4f}")


# =========================
# 7.3 定义模型和较小的参数搜索空间
# =========================
models_and_params = {
    "RandomForest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                random_state=random_seed,
                n_jobs=-1,
                class_weight="balanced"
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", 0.5]
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=random_seed,
                n_jobs=-1,
                scale_pos_weight=scale_pos_weight
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
        }
    },

    "MLP": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                random_state=random_seed,
                max_iter=1000,
                early_stopping=True
            ))
        ]),
        "param_grid": {
            "model__hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "model__alpha": [1e-4, 1e-3],
            "model__learning_rate_init": [1e-3, 5e-4]
        }
    },

    "LogisticRegression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=random_seed,
                class_weight="balanced",
                max_iter=2000
            ))
        ]),
        "param_grid": {
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear"]
        }
    }
}


# =========================
# 7.4 对每个模型进行 GridSearchCV
# =========================
best_estimators = {}
summary_results = []

for model_name, item in models_and_params.items():

    print("\n" + "=" * 80)
    print(f"Start Grid Search for: {model_name}")
    print("=" * 80)

    grid_search = GridSearchCV(
        estimator=item["pipeline"],
        param_grid=item["param_grid"],
        scoring=scoring,
        refit=refit_metric,
        cv=skf,
        n_jobs=-1,
        return_train_score=False
    )

    grid_search.fit(X, y)

    best_estimators[model_name] = grid_search.best_estimator_

    print(f"\nBest parameters for {model_name}:")
    print(grid_search.best_params_)

    print(f"\nBest mean CV {refit_metric}: {grid_search.best_score_:.4f}")

    best_idx = grid_search.best_index_

    result_row = {
        "Model": model_name,
        "Best_Params": grid_search.best_params_
    }

    for metric in scoring.keys():
        mean_score = grid_search.cv_results_[f"mean_test_{metric}"][best_idx]
        std_score = grid_search.cv_results_[f"std_test_{metric}"][best_idx]

        result_row[f"{metric}_mean"] = mean_score
        result_row[f"{metric}_std"] = std_score

        print(f"{metric:10s}: {mean_score:.4f} ± {std_score:.4f}")

    summary_results.append(result_row)


# =========================
# 7.5 汇总不同模型的调参结果
# =========================
summary_df = pd.DataFrame(summary_results)

display_cols = [
    "Model",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std",
    "roc_auc_mean", "roc_auc_std",
    "Best_Params"
]

summary_df = summary_df[display_cols]

print("\n" + "=" * 80)
print("Grid Search Summary")
print("=" * 80)
print(summary_df)


# =========================
# 8. 使用最佳模型进行整体交叉验证预测
# =========================
overall_results = []

for model_name, best_model in best_estimators.items():

    print("\n" + "=" * 80)
    print(f"Overall CV Prediction Metrics: {model_name}")
    print("=" * 80)

    y_pred = cross_val_predict(
        best_model,
        X,
        y,
        cv=skf,
        method="predict",
        n_jobs=-1
    )

    # 大多数模型都有 predict_proba
    if hasattr(best_model, "predict_proba"):
        y_proba = cross_val_predict(
            best_model,
            X,
            y,
            cv=skf,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
    else:
        y_proba = None

    acc = accuracy_score(y, y_pred)
    pre = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    if y_proba is not None:
        auc = roc_auc_score(y, y_proba)
    else:
        auc = np.nan

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y,
        y_pred,
        target_names=["weak", "strong"]
    ))

    overall_results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    })


# =========================
# 8.1 整体预测结果汇总
# =========================
overall_results_df = pd.DataFrame(overall_results)

print("\n" + "=" * 80)
print("Overall CV Prediction Summary")
print("=" * 80)
print(overall_results_df)


# =========================
# 8.2 选择最优模型
# =========================
best_model_name = overall_results_df.sort_values(
    by="F1",
    ascending=False
).iloc[0]["Model"]

best_model = best_estimators[best_model_name]

print("\n" + "=" * 80)
print(f"Best model based on overall CV F1-score: {best_model_name}")
print("=" * 80)

print("\nBest model pipeline:")
print(best_model)

df_md_best_overall = overall_results_df[
    overall_results_df["Model"] == best_model_name
].copy()

df_md_best_summary = pd.DataFrame({
    "method": ["MD"],
    "best_model": [best_model_name],
    "test_accuracy": [f"{df_md_best_overall['Accuracy'].iloc[0]:.4f}"],
    "test_precision": [f"{df_md_best_overall['Precision'].iloc[0]:.4f}"],
    "test_recall": [f"{df_md_best_overall['Recall'].iloc[0]:.4f}"],
    "test_f1": [f"{df_md_best_overall['F1'].iloc[0]:.4f}"],
    "test_roc_auc": [f"{df_md_best_overall['ROC_AUC'].iloc[0]:.4f}"]
})

df_PTT_best_summary = pd.concat([df_dft_md_best_summary, df_dft_best_summary, df_md_best_summary], ignore_index=True)

存在的特征列数: 6
缺失的列: []

最终用于建模的特征数: 6
最终特征列:
['vdw_energy_kjmol', 'total_energy_kjmol', 'rg_center_norm', 'rdf_first_peak_height', 'hbond_dynamic', 'hbond_static']

标签分布：
PTT_bin
1    129
0    114
Name: count, dtype: int64

===== 5-Fold Cross Validation Results =====
accuracy  : 0.6695 ± 0.1254
precision : 0.6743 ± 0.1061
recall    : 0.7301 ± 0.1615
f1        : 0.6972 ± 0.1240
roc_auc   : 0.7304 ± 0.1221

===== Overall CV Prediction Metrics =====
Accuracy : 0.6707818930041153
Precision: 0.6762589928057554
Recall   : 0.7286821705426356
F1-score : 0.7014925373134329
ROC-AUC  : 0.7280361757105943

===== Confusion Matrix =====
[[69 45]
 [35 94]]

===== Classification Report =====
              precision    recall  f1-score   support

        weak       0.66      0.61      0.63       114
      strong       0.68      0.73      0.70       129

    accuracy                           0.67       243
   macro avg       0.67      0.67      0.67       243
weighted avg       0.67      0.67      0.67   

In [13]:
df_PTT_best_summary

,method,best_model,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,DFT_MD,RandomForest,0.7737,0.7721,0.8140,0.7925,0.8365
1,DFT,RandomForest,0.7449,0.7481,0.7829,0.7652,0.8315
2,MD,RandomForest,0.6872,0.6963,0.7287,0.7121,0.7330


## 2. PDT

In [14]:
data_PDT["PDT_bin"] = data_PDT["photodynamic"].astype(str).replace({"middle": "weak"})
data_PDT['PDT_bin'].value_counts()

PDT_bin
strong    130
weak      113
Name: count, dtype: int64

### 2.1 DFT+MD

In [15]:
# =========================
# 1. 选择输入特征和标签
# =========================
feature_cols = DFT_cols + MD_cols

target_col = 'PDT_bin'

# 只保留实际存在的列
existing_feature_cols = [col for col in feature_cols if col in data_PDT.columns]
missing_cols = [col for col in feature_cols if col not in data_PDT.columns]

print("存在的特征列数:", len(existing_feature_cols))
print("缺失的列:", missing_cols)

# 提取数据
X = data_PDT[existing_feature_cols].copy()
y = data_PDT[target_col].copy()

# 2. 自动去掉非数值列
# =========================
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    print("\n以下列不是数值列，将被移除：")
    print(non_numeric_cols)

X = X.select_dtypes(include=[np.number]).copy()

print("\n最终用于建模的特征数:", X.shape[1])
print("最终特征列:")
print(X.columns.tolist())

# =========================
# 3. 标签编码
# strong -> 1, weak -> 0
# =========================
y = y.map({'strong': 1, 'weak': 0})

# 检查是否有未映射成功的标签
if y.isnull().any():
    raise ValueError("PDT_bin 中存在不是 'strong' 或 'weak' 的标签，请检查数据。")

print("\n标签分布：")
print(y.value_counts())
# =========================
# 4. 定义五折分层交叉验证
# =========================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_seed)

# =========================
# 5. 定义模型管道
# =========================
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=random_seed,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

# =========================
# 6. 交叉验证评估
# =========================
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    pipeline,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("\n===== 5-Fold Cross Validation Results =====")
for metric in scoring.keys():
    scores = cv_results[f'test_{metric}']
    print(f"{metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")

# =========================
# 7. 获取交叉验证整体预测结果
# =========================
y_pred = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict',
    n_jobs=-1
)

y_proba = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# =========================
# 8. 输出整体指标
# =========================
print("\n===== Overall CV Prediction Metrics =====")
print("Accuracy :", accuracy_score(y, y_pred))
print("Precision:", precision_score(y, y_pred))
print("Recall   :", recall_score(y, y_pred))
print("F1-score :", f1_score(y, y_pred))
print("ROC-AUC  :", roc_auc_score(y, y_proba))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y, y_pred))

print("\n===== Classification Report =====")
print(classification_report(y, y_pred, target_names=['weak', 'strong']))


################################### 调参 ########################################

# =========================
# 7. 模型调参 + 十折分层交叉验证
# =========================
# =========================
# 7.1 十折分层交叉验证设置
# =========================
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_seed
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# 调参时以 F1-score 作为主指标
refit_metric = "f1"


# =========================
# 7.2 类别不平衡处理
# =========================
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)

if n_pos == 0 or n_neg == 0:
    raise ValueError("y 中必须同时包含 strong 和 weak 两类。")

scale_pos_weight = n_neg / n_pos

print("\n类别分布:")
print(y.value_counts())
print(f"XGBoost scale_pos_weight: {scale_pos_weight:.4f}")


# =========================
# 7.3 定义模型和较小的参数搜索空间
# =========================
models_and_params = {
    "RandomForest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                random_state=random_seed,
                n_jobs=-1,
                class_weight="balanced"
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", 0.5]
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=random_seed,
                n_jobs=-1,
                scale_pos_weight=scale_pos_weight
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
        }
    },

    "MLP": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                random_state=random_seed,
                max_iter=1000,
                early_stopping=True
            ))
        ]),
        "param_grid": {
            "model__hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "model__alpha": [1e-4, 1e-3],
            "model__learning_rate_init": [1e-3, 5e-4]
        }
    },

    "LogisticRegression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=random_seed,
                class_weight="balanced",
                max_iter=2000
            ))
        ]),
        "param_grid": {
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear"]
        }
    }
}


# =========================
# 7.4 对每个模型进行 GridSearchCV
# =========================
best_estimators = {}
summary_results = []

for model_name, item in models_and_params.items():

    print("\n" + "=" * 80)
    print(f"Start Grid Search for: {model_name}")
    print("=" * 80)

    grid_search = GridSearchCV(
        estimator=item["pipeline"],
        param_grid=item["param_grid"],
        scoring=scoring,
        refit=refit_metric,
        cv=skf,
        n_jobs=-1,
        return_train_score=False
    )

    grid_search.fit(X, y)

    best_estimators[model_name] = grid_search.best_estimator_

    print(f"\nBest parameters for {model_name}:")
    print(grid_search.best_params_)

    print(f"\nBest mean CV {refit_metric}: {grid_search.best_score_:.4f}")

    best_idx = grid_search.best_index_

    result_row = {
        "Model": model_name,
        "Best_Params": grid_search.best_params_
    }

    for metric in scoring.keys():
        mean_score = grid_search.cv_results_[f"mean_test_{metric}"][best_idx]
        std_score = grid_search.cv_results_[f"std_test_{metric}"][best_idx]

        result_row[f"{metric}_mean"] = mean_score
        result_row[f"{metric}_std"] = std_score

        print(f"{metric:10s}: {mean_score:.4f} ± {std_score:.4f}")

    summary_results.append(result_row)


# =========================
# 7.5 汇总不同模型的调参结果
# =========================
summary_df = pd.DataFrame(summary_results)

display_cols = [
    "Model",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std",
    "roc_auc_mean", "roc_auc_std",
    "Best_Params"
]

summary_df = summary_df[display_cols]

print("\n" + "=" * 80)
print("Grid Search Summary")
print("=" * 80)
print(summary_df)


# =========================
# 8. 使用最佳模型进行整体交叉验证预测
# =========================
overall_results = []

for model_name, best_model in best_estimators.items():

    print("\n" + "=" * 80)
    print(f"Overall CV Prediction Metrics: {model_name}")
    print("=" * 80)

    y_pred = cross_val_predict(
        best_model,
        X,
        y,
        cv=skf,
        method="predict",
        n_jobs=-1
    )

    # 大多数模型都有 predict_proba
    if hasattr(best_model, "predict_proba"):
        y_proba = cross_val_predict(
            best_model,
            X,
            y,
            cv=skf,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
    else:
        y_proba = None

    acc = accuracy_score(y, y_pred)
    pre = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    if y_proba is not None:
        auc = roc_auc_score(y, y_proba)
    else:
        auc = np.nan

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y,
        y_pred,
        target_names=["weak", "strong"]
    ))

    overall_results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    })


# =========================
# 8.1 整体预测结果汇总
# =========================
overall_results_df = pd.DataFrame(overall_results)

print("\n" + "=" * 80)
print("Overall CV Prediction Summary")
print("=" * 80)
print(overall_results_df)


# =========================
# 8.2 选择最优模型
# =========================
best_model_name = overall_results_df.sort_values(
    by="F1",
    ascending=False
).iloc[0]["Model"]

best_model = best_estimators[best_model_name]

print("\n" + "=" * 80)
print(f"Best model based on overall CV F1-score: {best_model_name}")
print("=" * 80)

print("\nBest model pipeline:")
print(best_model)

df_PDT_dft_md_best_overall = overall_results_df[
    overall_results_df["Model"] == best_model_name
].copy()

df_PDT_dft_md_best_summary = pd.DataFrame({
    "method": ["DFT_MD"],
    "best_model": [best_model_name],
    "test_accuracy": [f"{df_PDT_dft_md_best_overall['Accuracy'].iloc[0]:.4f}"],
    "test_precision": [f"{df_PDT_dft_md_best_overall['Precision'].iloc[0]:.4f}"],
    "test_recall": [f"{df_PDT_dft_md_best_overall['Recall'].iloc[0]:.4f}"],
    "test_f1": [f"{df_PDT_dft_md_best_overall['F1'].iloc[0]:.4f}"],
    "test_roc_auc": [f"{df_PDT_dft_md_best_overall['ROC_AUC'].iloc[0]:.4f}"]
})

存在的特征列数: 21
缺失的列: []

最终用于建模的特征数: 21
最终特征列:
['es1_ev_high_pred', 'fs1_high_pred', 'et1_ev_high_pred', 'delta_est_ev_high_pred', 'min_abs_s1_tn_ev_high_pred', 'd_index_ang_high_pred', 't_index_ang_high_pred', 'zpe_ha_pred', 'gcorr_ha_pred', 'alpha_iso_bohr3_pred', 's_vib_cal_molk_pred', 'svib_frac_pred', 'hlg_kjmol_high_pred', 'dipole_gs_debye_high_pred', 'ci_index', 'vdw_energy_kjmol', 'total_energy_kjmol', 'rg_center_norm', 'rdf_first_peak_height', 'hbond_dynamic', 'hbond_static']

标签分布：
PDT_bin
1    130
0    113
Name: count, dtype: int64

===== 5-Fold Cross Validation Results =====
accuracy  : 0.7202 ± 0.0981
precision : 0.7514 ± 0.0988
recall    : 0.7231 ± 0.1250
f1        : 0.7320 ± 0.0960
roc_auc   : 0.8085 ± 0.0797

===== Overall CV Prediction Metrics =====
Accuracy : 0.720164609053498
Precision: 0.746031746031746
Recall   : 0.7230769230769231
F1-score : 0.734375
ROC-AUC  : 0.8014976174268209

===== Confusion Matrix =====
[[81 32]
 [36 94]]

===== Classification Report =====
    

### 2.2 DFT

In [16]:
########################################### DFT #####################################################

# =========================
# 1. 选择输入特征和标签
# =========================
feature_cols = DFT_cols

target_col = 'PDT_bin'

# 只保留实际存在的列
existing_feature_cols = [col for col in feature_cols if col in data_PDT.columns]
missing_cols = [col for col in feature_cols if col not in data_PDT.columns]

print("存在的特征列数:", len(existing_feature_cols))
print("缺失的列:", missing_cols)

# 提取数据
X = data_PDT[existing_feature_cols].copy()
y = data_PDT[target_col].copy()

# 2. 自动去掉非数值列
# =========================
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    print("\n以下列不是数值列，将被移除：")
    print(non_numeric_cols)

X = X.select_dtypes(include=[np.number]).copy()

print("\n最终用于建模的特征数:", X.shape[1])
print("最终特征列:")
print(X.columns.tolist())

# =========================
# 3. 标签编码
# strong -> 1, weak -> 0
# =========================
y = y.map({'strong': 1, 'weak': 0})

# 检查是否有未映射成功的标签
if y.isnull().any():
    raise ValueError("PDT_bin 中存在不是 'strong' 或 'weak' 的标签，请检查数据。")

print("\n标签分布：")
print(y.value_counts())
# =========================
# 4. 定义五折分层交叉验证
# =========================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_seed)

# =========================
# 5. 定义模型管道
# =========================
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=random_seed,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

# =========================
# 6. 交叉验证评估
# =========================
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    pipeline,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("\n===== 5-Fold Cross Validation Results =====")
for metric in scoring.keys():
    scores = cv_results[f'test_{metric}']
    print(f"{metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")

# =========================
# 7. 获取交叉验证整体预测结果
# =========================
y_pred = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict',
    n_jobs=-1
)

y_proba = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# =========================
# 8. 输出整体指标
# =========================
print("\n===== Overall CV Prediction Metrics =====")
print("Accuracy :", accuracy_score(y, y_pred))
print("Precision:", precision_score(y, y_pred))
print("Recall   :", recall_score(y, y_pred))
print("F1-score :", f1_score(y, y_pred))
print("ROC-AUC  :", roc_auc_score(y, y_proba))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y, y_pred))

print("\n===== Classification Report =====")
print(classification_report(y, y_pred, target_names=['weak', 'strong']))


#################################### 调参 ########################################

# =========================
# 7. 模型调参 + 十折分层交叉验证
# =========================
# =========================
# 7.1 十折分层交叉验证设置
# =========================
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_seed
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# 调参时以 F1-score 作为主指标
refit_metric = "f1"


# =========================
# 7.2 类别不平衡处理
# =========================
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)

if n_pos == 0 or n_neg == 0:
    raise ValueError("y 中必须同时包含 strong 和 weak 两类。")

scale_pos_weight = n_neg / n_pos

print("\n类别分布:")
print(y.value_counts())
print(f"XGBoost scale_pos_weight: {scale_pos_weight:.4f}")


# =========================
# 7.3 定义模型和较小的参数搜索空间
# =========================
models_and_params = {
    "RandomForest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                random_state=random_seed,
                n_jobs=-1,
                class_weight="balanced"
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", 0.5]
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=random_seed,
                n_jobs=-1,
                scale_pos_weight=scale_pos_weight
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
        }
    },

    "MLP": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                random_state=random_seed,
                max_iter=1000,
                early_stopping=True
            ))
        ]),
        "param_grid": {
            "model__hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "model__alpha": [1e-4, 1e-3],
            "model__learning_rate_init": [1e-3, 5e-4]
        }
    },

    "LogisticRegression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=random_seed,
                class_weight="balanced",
                max_iter=2000
            ))
        ]),
        "param_grid": {
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear"]
        }
    }
}


# =========================
# 7.4 对每个模型进行 GridSearchCV
# =========================
best_estimators = {}
summary_results = []

for model_name, item in models_and_params.items():

    print("\n" + "=" * 80)
    print(f"Start Grid Search for: {model_name}")
    print("=" * 80)

    grid_search = GridSearchCV(
        estimator=item["pipeline"],
        param_grid=item["param_grid"],
        scoring=scoring,
        refit=refit_metric,
        cv=skf,
        n_jobs=-1,
        return_train_score=False
    )

    grid_search.fit(X, y)

    best_estimators[model_name] = grid_search.best_estimator_

    print(f"\nBest parameters for {model_name}:")
    print(grid_search.best_params_)

    print(f"\nBest mean CV {refit_metric}: {grid_search.best_score_:.4f}")

    best_idx = grid_search.best_index_

    result_row = {
        "Model": model_name,
        "Best_Params": grid_search.best_params_
    }

    for metric in scoring.keys():
        mean_score = grid_search.cv_results_[f"mean_test_{metric}"][best_idx]
        std_score = grid_search.cv_results_[f"std_test_{metric}"][best_idx]

        result_row[f"{metric}_mean"] = mean_score
        result_row[f"{metric}_std"] = std_score

        print(f"{metric:10s}: {mean_score:.4f} ± {std_score:.4f}")

    summary_results.append(result_row)


# =========================
# 7.5 汇总不同模型的调参结果
# =========================
summary_df = pd.DataFrame(summary_results)

display_cols = [
    "Model",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std",
    "roc_auc_mean", "roc_auc_std",
    "Best_Params"
]

summary_df = summary_df[display_cols]

print("\n" + "=" * 80)
print("Grid Search Summary")
print("=" * 80)
print(summary_df)


# =========================
# 8. 使用最佳模型进行整体交叉验证预测
# =========================
overall_results = []

for model_name, best_model in best_estimators.items():

    print("\n" + "=" * 80)
    print(f"Overall CV Prediction Metrics: {model_name}")
    print("=" * 80)

    y_pred = cross_val_predict(
        best_model,
        X,
        y,
        cv=skf,
        method="predict",
        n_jobs=-1
    )

    # 大多数模型都有 predict_proba
    if hasattr(best_model, "predict_proba"):
        y_proba = cross_val_predict(
            best_model,
            X,
            y,
            cv=skf,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
    else:
        y_proba = None

    acc = accuracy_score(y, y_pred)
    pre = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    if y_proba is not None:
        auc = roc_auc_score(y, y_proba)
    else:
        auc = np.nan

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y,
        y_pred,
        target_names=["weak", "strong"]
    ))

    overall_results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    })


# =========================
# 8.1 整体预测结果汇总
# =========================
overall_results_df = pd.DataFrame(overall_results)

print("\n" + "=" * 80)
print("Overall CV Prediction Summary")
print("=" * 80)
print(overall_results_df)


# =========================
# 8.2 选择最优模型
# =========================
best_model_name = overall_results_df.sort_values(
    by="F1",
    ascending=False
).iloc[0]["Model"]

best_model = best_estimators[best_model_name]

print("\n" + "=" * 80)
print(f"Best model based on overall CV F1-score: {best_model_name}")
print("=" * 80)

print("\nBest model pipeline:")
print(best_model)

df_PDT_dft_best_overall = overall_results_df[
    overall_results_df["Model"] == best_model_name
].copy()

df_PDT_dft_best_summary = pd.DataFrame({
    "method": ["DFT"],
    "best_model": [best_model_name],
    "test_accuracy": [f"{df_PDT_dft_best_overall['Accuracy'].iloc[0]:.4f}"],
    "test_precision": [f"{df_PDT_dft_best_overall['Precision'].iloc[0]:.4f}"],
    "test_recall": [f"{df_PDT_dft_best_overall['Recall'].iloc[0]:.4f}"],
    "test_f1": [f"{df_PDT_dft_best_overall['F1'].iloc[0]:.4f}"],
    "test_roc_auc": [f"{df_PDT_dft_best_overall['ROC_AUC'].iloc[0]:.4f}"]
})

df_PDT_best_summary = pd.concat([df_PDT_dft_md_best_summary, df_PDT_dft_best_summary], ignore_index=True)

存在的特征列数: 15
缺失的列: []

最终用于建模的特征数: 15
最终特征列:
['es1_ev_high_pred', 'fs1_high_pred', 'et1_ev_high_pred', 'delta_est_ev_high_pred', 'min_abs_s1_tn_ev_high_pred', 'd_index_ang_high_pred', 't_index_ang_high_pred', 'zpe_ha_pred', 'gcorr_ha_pred', 'alpha_iso_bohr3_pred', 's_vib_cal_molk_pred', 'svib_frac_pred', 'hlg_kjmol_high_pred', 'dipole_gs_debye_high_pred', 'ci_index']

标签分布：
PDT_bin
1    130
0    113
Name: count, dtype: int64

===== 5-Fold Cross Validation Results =====
accuracy  : 0.7243 ± 0.0905
precision : 0.7518 ± 0.0933
recall    : 0.7308 ± 0.0988
f1        : 0.7387 ± 0.0854
roc_auc   : 0.8168 ± 0.0755

===== Overall CV Prediction Metrics =====
Accuracy : 0.7242798353909465
Precision: 0.7480314960629921
Recall   : 0.7307692307692307
F1-score : 0.7392996108949417
ROC-AUC  : 0.8103471749489447

===== Confusion Matrix =====
[[81 32]
 [35 95]]

===== Classification Report =====
              precision    recall  f1-score   support

        weak       0.70      0.72      0.71       113
 

### 2.3 MD

In [ ]:
########################################### MD #####################################################

# =========================
# 1. 选择输入特征和标签
# =========================
feature_cols = MD_cols

target_col = 'PDT_bin'

# 只保留实际存在的列
existing_feature_cols = [col for col in feature_cols if col in data_PDT.columns]
missing_cols = [col for col in feature_cols if col not in data_PDT.columns]

print("存在的特征列数:", len(existing_feature_cols))
print("缺失的列:", missing_cols)

# 提取数据
X = data_PDT[existing_feature_cols].copy()
y = data_PDT[target_col].copy()

# 2. 自动去掉非数值列
# =========================
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    print("\n以下列不是数值列，将被移除：")
    print(non_numeric_cols)

X = X.select_dtypes(include=[np.number]).copy()

print("\n最终用于建模的特征数:", X.shape[1])
print("最终特征列:")
print(X.columns.tolist())

# =========================
# 3. 标签编码
# strong -> 1, weak -> 0
# =========================
y = y.map({'strong': 1, 'weak': 0})

# 检查是否有未映射成功的标签
if y.isnull().any():
    raise ValueError("PDT_bin 中存在不是 'strong' 或 'weak' 的标签，请检查数据。")

print("\n标签分布:")
print(y.value_counts())
# =========================
# 4. 定义五折分层交叉验证
# =========================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_seed)

# =========================
# 5. 定义模型管道
# =========================
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=random_seed,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

# =========================
# 6. 交叉验证评估
# =========================
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    pipeline,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("\n===== 10-Fold Cross Validation Results =====")
for metric in scoring.keys():
    scores = cv_results[f'test_{metric}']
    print(f"{metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")

# =========================
# 7. 获取交叉验证整体预测结果
# =========================
y_pred = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict',
    n_jobs=-1
)

y_proba = cross_val_predict(
    pipeline,
    X,
    y,
    cv=skf,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# =========================
# 8. 输出整体指标
# =========================
print("\n===== Overall CV Prediction Metrics =====")
print("Accuracy :", accuracy_score(y, y_pred))
print("Precision:", precision_score(y, y_pred))
print("Recall   :", recall_score(y, y_pred))
print("F1-score :", f1_score(y, y_pred))
print("ROC-AUC  :", roc_auc_score(y, y_proba))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y, y_pred))

print("\n===== Classification Report =====")
print(classification_report(y, y_pred, target_names=['weak', 'strong']))


#################################### 调参 ########################################

# =========================
# 7. 模型调参 + 十折分层交叉验证
# =========================
# =========================
# 7.1 十折分层交叉验证设置
# =========================
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_seed
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# 调参时以 F1-score 作为主指标
refit_metric = "f1"


# =========================
# 7.2 类别不平衡处理
# =========================
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)

if n_pos == 0 or n_neg == 0:
    raise ValueError("y 中必须同时包含 strong 和 weak 两类。")

scale_pos_weight = n_neg / n_pos

print("\n类别分布:")
print(y.value_counts())
print(f"XGBoost scale_pos_weight: {scale_pos_weight:.4f}")


# =========================
# 7.3 定义模型和较小的参数搜索空间
# =========================
models_and_params = {
    "RandomForest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                random_state=random_seed,
                n_jobs=-1,
                class_weight="balanced"
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", 0.5]
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=random_seed,
                n_jobs=-1,
                scale_pos_weight=scale_pos_weight
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
        }
    },

    "MLP": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                random_state=random_seed,
                max_iter=1000,
                early_stopping=True
            ))
        ]),
        "param_grid": {
            "model__hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "model__alpha": [1e-4, 1e-3],
            "model__learning_rate_init": [1e-3, 5e-4]
        }
    },

    "LogisticRegression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=random_seed,
                class_weight="balanced",
                max_iter=2000
            ))
        ]),
        "param_grid": {
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear"]
        }
    }
}


# =========================
# 7.4 对每个模型进行 GridSearchCV
# =========================
best_estimators = {}
summary_results = []

for model_name, item in models_and_params.items():

    print("\n" + "=" * 80)
    print(f"Start Grid Search for: {model_name}")
    print("=" * 80)

    grid_search = GridSearchCV(
        estimator=item["pipeline"],
        param_grid=item["param_grid"],
        scoring=scoring,
        refit=refit_metric,
        cv=skf,
        n_jobs=-1,
        return_train_score=False
    )

    grid_search.fit(X, y)

    best_estimators[model_name] = grid_search.best_estimator_

    print(f"\nBest parameters for {model_name}:")
    print(grid_search.best_params_)

    print(f"\nBest mean CV {refit_metric}: {grid_search.best_score_:.4f}")

    best_idx = grid_search.best_index_

    result_row = {
        "Model": model_name,
        "Best_Params": grid_search.best_params_
    }

    for metric in scoring.keys():
        mean_score = grid_search.cv_results_[f"mean_test_{metric}"][best_idx]
        std_score = grid_search.cv_results_[f"std_test_{metric}"][best_idx]

        result_row[f"{metric}_mean"] = mean_score
        result_row[f"{metric}_std"] = std_score

        print(f"{metric:10s}: {mean_score:.4f} ± {std_score:.4f}")

    summary_results.append(result_row)


# =========================
# 7.5 汇总不同模型的调参结果
# =========================
summary_df = pd.DataFrame(summary_results)

display_cols = [
    "Model",
    "accuracy_mean", "accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "f1_mean", "f1_std",
    "roc_auc_mean", "roc_auc_std",
    "Best_Params"
]

summary_df = summary_df[display_cols]

print("\n" + "=" * 80)
print("Grid Search Summary")
print("=" * 80)
print(summary_df)


# =========================
# 8. 使用最佳模型进行整体交叉验证预测
# =========================
overall_results = []

for model_name, best_model in best_estimators.items():

    print("\n" + "=" * 80)
    print(f"Overall CV Prediction Metrics: {model_name}")
    print("=" * 80)

    y_pred = cross_val_predict(
        best_model,
        X,
        y,
        cv=skf,
        method="predict",
        n_jobs=-1
    )

    # 大多数模型都有 predict_proba
    if hasattr(best_model, "predict_proba"):
        y_proba = cross_val_predict(
            best_model,
            X,
            y,
            cv=skf,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
    else:
        y_proba = None

    acc = accuracy_score(y, y_pred)
    pre = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    if y_proba is not None:
        auc = roc_auc_score(y, y_proba)
    else:
        auc = np.nan

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y,
        y_pred,
        target_names=["weak", "strong"]
    ))

    overall_results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    })


# =========================
# 8.1 整体预测结果汇总
# =========================
overall_results_df = pd.DataFrame(overall_results)

print("\n" + "=" * 80)
print("Overall CV Prediction Summary")
print("=" * 80)
print(overall_results_df)


# =========================
# 8.2 选择最优模型
# =========================
best_model_name = overall_results_df.sort_values(
    by="F1",
    ascending=False
).iloc[0]["Model"]

best_model = best_estimators[best_model_name]

print("\n" + "=" * 80)
print(f"Best model based on overall CV F1-score: {best_model_name}")
print("=" * 80)

print("\nBest model pipeline:")
print(best_model)

df_PDT_md_best_overall = overall_results_df[
    overall_results_df["Model"] == best_model_name
].copy()

df_PDT_md_best_summary = pd.DataFrame({
    "method": ["MD"],
    "best_model": [best_model_name],
    "test_accuracy": [f"{df_PDT_md_best_overall['Accuracy'].iloc[0]:.4f}"],
    "test_precision": [f"{df_PDT_md_best_overall['Precision'].iloc[0]:.4f}"],
    "test_recall": [f"{df_PDT_md_best_overall['Recall'].iloc[0]:.4f}"],
    "test_f1": [f"{df_PDT_md_best_overall['F1'].iloc[0]:.4f}"],
    "test_roc_auc": [f"{df_PDT_md_best_overall['ROC_AUC'].iloc[0]:.4f}"]
})

df_PDT_best_summary = pd.concat([df_PDT_dft_md_best_summary, df_PDT_md_best_summary, df_PDT_dft_best_summary], ignore_index=True)

存在的特征列数: 6
缺失的列: []

最终用于建模的特征数: 6
最终特征列:
['vdw_energy_kjmol', 'total_energy_kjmol', 'rg_center_norm', 'rdf_first_peak_height', 'hbond_dynamic', 'hbond_static']

标签分布:
PDT_bin
1    130
0    113
Name: count, dtype: int64

===== 10-Fold Cross Validation Results =====
accuracy  : 0.6245 ± 0.0944
precision : 0.6624 ± 0.0814
recall    : 0.6154 ± 0.1418
f1        : 0.6318 ± 0.0988
roc_auc   : 0.6820 ± 0.1084

===== Overall CV Prediction Metrics =====
Accuracy : 0.6255144032921811
Precision: 0.6611570247933884
Recall   : 0.6153846153846154
F1-score : 0.6374501992031872
ROC-AUC  : 0.6780122532334921

===== Confusion Matrix =====
[[72 41]
 [50 80]]

===== Classification Report =====
              precision    recall  f1-score   support

        weak       0.59      0.64      0.61       113
      strong       0.66      0.62      0.64       130

    accuracy                           0.63       243
   macro avg       0.63      0.63      0.63       243
weighted avg       0.63      0.63      0.63  

In [19]:
df_PDT_best_summary 

,method,best_model,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,DFT_MD,RandomForest,0.7449,0.7656,0.7538,0.7597,0.8049
1,MD,MLP,0.5350,0.5350,1.0000,0.6971,0.4894
2,DFT,RandomForest,0.7407,0.7597,0.7538,0.7568,0.8097
